Ce code montre les étapes principales d’un GAN :

Charger les données

Définir le générateur

Définir le discriminateur

Entraîner les deux réseaux en compétition

Générer de nouvelles images

In [ ]:
# ============================================
# GAN simple avec PyTorch
# ============================================

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt


# ============================================
# Paramètres
# ============================================

batch_size = 64
latent_dim = 100
epochs = 10
lr = 0.0002


# ============================================
# Dataset MNIST
# ============================================

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    transform=transform,
    download=True
)

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True
)


# ============================================
# Générateur
# ============================================

class Generator(nn.Module):

    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(True),

            nn.Linear(256, 512),
            nn.ReLU(True),

            nn.Linear(512, 784),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)


# ============================================
# Discriminateur
# ============================================

class Discriminator(nn.Module):

    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),

            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)


# ============================================
# Initialisation
# ============================================

generator = Generator()
discriminator = Discriminator()

criterion = nn.BCELoss()

optimizer_G = optim.Adam(generator.parameters(), lr=lr)
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr)


# ============================================
# Entraînement
# ============================================

for epoch in range(epochs):

    for real_images, _ in dataloader:

        batch_size_current = real_images.size(0)
        real_images = real_images.view(batch_size_current, 784)

        real_labels = torch.ones(batch_size_current, 1)
        fake_labels = torch.zeros(batch_size_current, 1)

        # -------------------------
        # Train Discriminator
        # -------------------------

        outputs = discriminator(real_images)
        loss_real = criterion(outputs, real_labels)

        noise = torch.randn(batch_size_current, latent_dim)
        fake_images = generator(noise)

        outputs = discriminator(fake_images.detach())
        loss_fake = criterion(outputs, fake_labels)

        loss_D = loss_real + loss_fake

        optimizer_D.zero_grad()
        loss_D.backward()
        optimizer_D.step()

        # -------------------------
        # Train Generator
        # -------------------------

        noise = torch.randn(batch_size_current, latent_dim)
        fake_images = generator(noise)

        outputs = discriminator(fake_images)
        loss_G = criterion(outputs, real_labels)

        optimizer_G.zero_grad()
        loss_G.backward()
        optimizer_G.step()

    print(f"Epoch [{epoch+1}/{epochs}]  Loss_D: {loss_D.item():.4f}  Loss_G: {loss_G.item():.4f}")


# ============================================
# Génération d'images
# ============================================

noise = torch.randn(16, latent_dim)
generated_images = generator(noise).detach().numpy()

fig, axes = plt.subplots(4,4, figsize=(6,6))

for i, ax in enumerate(axes.flat):
    ax.imshow(generated_images[i].reshape(28,28), cmap='gray')
    ax.axis('off')

plt.show()